# Module 5 — Delta Live Tables (DLT)
Exam domain: **Data Processing**

Databricks: this cell content is meant to be saved as a `.py` source file and
attached to a **DLT pipeline** (Workflows -> Delta Live Tables -> Create
Pipeline -> point it at this file). It will not execute as a normal interactive
notebook run — DLT pipelines have their own managed runtime.

In [ ]:
import dlt
from pyspark.sql import functions as F

@dlt.table(
    name="bronze_events",
    comment="Raw events, ingested as-is via Auto Loader"
)
def bronze_events():
    return (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/Volumes/main/module5/_schema")
        .load("/Volumes/main/module5/landing"))

In [ ]:
@dlt.table(
    name="silver_events",
    comment="Cleaned, validated events"
)
@dlt.expect_or_drop("valid_amount", "amount IS NOT NULL AND amount > 0")
@dlt.expect("valid_name", "name IS NOT NULL")
def silver_events():
    return (dlt.read_stream("bronze_events")
        .withColumn("amount", F.col("amount_raw").cast("double"))
        .withColumn("event_date", F.to_date("event_date"))
        .drop("amount_raw"))

In [ ]:
@dlt.table(
    name="gold_daily_summary",
    comment="Daily aggregates for BI"
)
def gold_daily_summary():
    return (dlt.read("silver_events")
        .groupBy("event_date")
        .agg(F.sum("amount").alias("total_amount"), F.count("*").alias("num_events")))

## Creating the pipeline
1. Save the three cells above as `dlt_pipeline.py` in a Databricks Repo.
2. Workflows -> Delta Live Tables -> Create Pipeline -> source = that file.
3. Choose **Triggered** (batch-like, matches Module 4's `availableNow`) or
   **Continuous**.
4. Run the pipeline and inspect the DAG it infers automatically from the
   `dlt.read_stream("bronze_events")` / `dlt.read("silver_events")` calls.

## Monitoring expectations
The pipeline's **Data Quality** tab reports, per table, how many rows passed vs.
were dropped by each `@dlt.expect*` rule — this is the DLT-native equivalent of
the manual row-count checks you'd otherwise write by hand.